# Illness Report Analysis
**Checkpoint:** `matryoshka/layer-3/xray-conglomerate/xray_v1/step_0200000`  
**252 confirmed universal latents** across CheXpert, RSNA, SIIM-COVID, TB

In [ ]:
import csv
import json
import numpy as np
from collections import defaultdict
from pathlib import Path

BASE = Path('.')  # run from the ilness-report directory

# Load all files
with open(BASE / 'illness_eval_summary.json') as f:
    eval_summary = json.load(f)

with open(BASE / 'illness_latent_summary.json') as f:
    latent_summary = json.load(f)

with open(BASE / 'illness_latent_matrix.csv') as f:
    matrix_rows = list(csv.DictReader(f))

with open(BASE / 'illness_discriminability_probe_coarse.csv') as f:
    coarse_rows = list(csv.DictReader(f))

with open(BASE / 'illness_discriminability_probe_fine.csv') as f:
    fine_rows = list(csv.DictReader(f))

with open(BASE / 'illness_activation_profiles.csv') as f:
    profile_rows = list(csv.DictReader(f))

with open(BASE / 'illness_spatial_grounding.csv') as f:
    sg_rows = list(csv.DictReader(f))

print(f"Eval summary: {eval_summary['n_confirmed_latents']} confirmed latents, {eval_summary['n_patches']} patches")
print(f"Datasets: {eval_summary['datasets']}")

## 1. Latent Matrix — Illness Groups × Confirmed Latents

In [ ]:
illness_groups = [r['illness_group'] for r in matrix_rows]
latent_cols = [c for c in matrix_rows[0].keys() if c != 'illness_group']
matrix = np.array([[float(r[c]) for c in latent_cols] for r in matrix_rows])

print(f"Matrix shape: {matrix.shape}  (illness_groups × latents)")
print(f"Illness groups: {illness_groups}")
print(f"Activation range: {matrix.min():.1f} – {matrix.max():.1f}")

## 2. Top Latents by Raw Activation vs. by Specificity

In [ ]:
mean_act = matrix.mean(axis=0)
max_act = matrix.max(axis=0)
specificity = max_act / (mean_act + 1e-6)  # max/mean ratio

print("=== Top 10 latents by RAW mean activation (misleading — dominated by backbone features) ===")
for i in np.argsort(mean_act)[::-1][:10]:
    print(f"  latent {latent_cols[i]:>5}: mean={mean_act[i]:.1f}")

print()
print("=== Top 10 latents by ILLNESS SPECIFICITY (max/mean ratio) ===")
for i in np.argsort(specificity)[::-1][:10]:
    best_ill = np.argmax(matrix[:, i])
    print(f"  latent {latent_cols[i]:>5}: specificity={specificity[i]:.2f}, peaks at [{illness_groups[best_ill]}] (act={matrix[best_ill,i]:.1f})")

## 3. Standout Latents per Illness Group (z-score relative to other groups)

In [ ]:
print("=== Per-illness standout latents (z-score > 2 relative to other illness groups) ===")
for ill_i, ill_name in enumerate(illness_groups):
    row = matrix[ill_i]
    other_rows = np.delete(matrix, ill_i, axis=0)
    diff = row - other_rows.mean(axis=0)
    threshold = diff.mean() + 2 * diff.std()
    standouts = [(latent_cols[j], row[j], diff[j]) for j in range(len(latent_cols)) if diff[j] > threshold]
    standouts.sort(key=lambda x: -x[2])
    print(f"  {ill_name}:")
    for lat, act, delta in standouts[:5]:
        print(f"    latent {lat:>5}  act={act:.1f}  Δ={delta:.1f}")

## 4. Universal Backbone Latents — Appear in Top-10 for All 7 Illness Groups

In [ ]:
shared_count = {}
for ill, latents in latent_summary['top_k_latents_per_illness'].items():
    for l in latents:
        shared_count[l] = shared_count.get(l, 0) + 1

print("Latents in top-10 of ALL 7 illness groups (backbone / non-discriminative):")
print([k for k, v in shared_count.items() if v == 7])

print()
print("=== Latent 3512 activation per illness (activation_freq from profiles) ===")
for special in ['3512', '2640']:
    col_i = latent_cols.index(special)
    print(f"Latent {special}:")
    for ill_i, ill_name in enumerate(illness_groups):
        print(f"  {ill_name:<25}: {matrix[ill_i, col_i]:.1f}")

## 5. Coarse Discriminability (AUROC) per Dataset

In [ ]:
by_dataset = defaultdict(list)
for r in coarse_rows:
    if r['auroc_mean'] != 'nan':
        by_dataset[r['dataset']].append(
            (float(r['auroc_mean']), r['latent_idx'], float(r['auroc_std']))
        )

for ds, entries in by_dataset.items():
    entries.sort(reverse=True)
    print(f"\n=== Top 10 latents by AUROC: {ds} (n={len(entries)} valid) ===")
    for auroc, lat, std in entries[:10]:
        print(f"  latent {lat:>5}: AUROC={auroc:.4f} ± {std:.4f}")

nan_datasets = [ds for ds, entries in [(r['dataset'], r['auroc_mean']) for r in coarse_rows] if False]
all_datasets = set(r['dataset'] for r in coarse_rows)
valid_datasets = set(by_dataset.keys())
print(f"\nDatasets with all-NaN AUROC: {all_datasets - valid_datasets}")

## 6. Activation Profiles for Key Discriminating Latents

In [ ]:
# Build profile lookup: latent -> illness_group -> dataset -> (mean_act, freq, n)
profiles = defaultdict(lambda: defaultdict(dict))
for r in profile_rows:
    profiles[r['latent_idx']][r['illness_group']][r['dataset']] = (
        float(r['mean_activation']), float(r['activation_freq']), int(r['n_images'])
    )

# Key latents: healthy-lung detectors and pathology detectors
key_latents = {
    '818':  'Healthy-lung detector',
    '1384': 'Healthy-lung detector (strongest AUROC)',
    '884':  'Healthy-lung detector (secondary)',
    '2682': 'Pathology-presence detector (airspace/pleural)',
    '3348': 'Pathology-presence detector (pleural/fluid)',
    '2619': 'Pathology-presence detector (cardiac/vascular)',
    '3512': 'Backbone (always-on)',
    '2640': 'Backbone (always-on)',
}

for latent, role in key_latents.items():
    print(f"\n=== Latent {latent} — {role} ===")
    print(f"  {'illness':<25} {'chexpert act|freq':>20} {'siim_covid act|freq':>22} {'tb act|freq':>15}")
    for ill in illness_groups:
        parts = []
        for ds in ['chexpert', 'siim_covid', 'tb']:
            if ds in profiles[latent][ill]:
                act, freq, n = profiles[latent][ill][ds]
                parts.append(f"{act:6.1f}|{freq:.2f}")
            else:
                parts.append('         —')
        print(f"  {ill:<25} {parts[0]:>20} {parts[1]:>22} {parts[2]:>15}")

## 7. Fine-Grained Label Probe and Spatial Grounding — Data Gap Check

In [ ]:
# Fine probe NaN analysis
fine_nan = sum(1 for r in fine_rows if r['auroc_mean'] == 'nan')
fine_total = len(fine_rows)
fine_datasets = set(r['dataset'] for r in fine_rows)
fine_labels = sorted(set(r['label_name'] for r in fine_rows))

print(f"Fine probe: {fine_nan}/{fine_total} rows are NaN ({100*fine_nan/fine_total:.1f}%)")
print(f"Fine probe datasets: {fine_datasets}")
print(f"Fine probe labels: {fine_labels}")

# Non-NaN fine probe entries
fine_valid = [(float(r['auroc_mean']), r['latent_idx'], r['label_name'], r['dataset'])
               for r in fine_rows if r['auroc_mean'] != 'nan']
fine_valid.sort(reverse=True)
print(f"\nNon-NaN fine probe entries: {len(fine_valid)}")
if fine_valid:
    print("Top 5:")
    for auroc, lat, label, ds in fine_valid[:5]:
        print(f"  latent {lat}, {label} ({ds}): AUROC={auroc:.4f}")

print()

# Spatial grounding NaN analysis
sg_nan = sum(1 for r in sg_rows if r['pearson_r'] == 'nan')
sg_total = len(sg_rows)
sg_nonzero_images = [r for r in sg_rows if int(r['n_images']) > 0]
print(f"Spatial grounding: {sg_nan}/{sg_total} rows are NaN ({100*sg_nan/sg_total:.1f}%)")
print(f"Entries with n_images > 0 but still NaN: {len(sg_nonzero_images)}")
print("=> PIPELINE BUG: spatial correlation not computed despite images being present")

## 8. Summary Table — Top Discriminating Latents

In [ ]:
# Build per-latent AUROC across datasets
auroc_by_latent = defaultdict(dict)
for r in coarse_rows:
    if r['auroc_mean'] != 'nan':
        auroc_by_latent[r['latent_idx']][r['dataset']] = float(r['auroc_mean'])

# Score by mean AUROC across available datasets (excluding RSNA which is all NaN)
scored = []
for lat, ds_dict in auroc_by_latent.items():
    vals = list(ds_dict.values())
    scored.append((np.mean(vals), lat, ds_dict))
scored.sort(reverse=True)

print(f"{'Rank':>4} {'Latent':>7} {'Mean AUROC':>12} {'CheXpert':>10} {'SIIM-COVID':>12} {'TB':>8}")
print('-' * 60)
for rank, (mean_auroc, lat, ds_dict) in enumerate(scored[:20], 1):
    cx = ds_dict.get('chexpert', float('nan'))
    sc = ds_dict.get('siim_covid', float('nan'))
    tb = ds_dict.get('tb', float('nan'))
    print(f"{rank:>4} {lat:>7} {mean_auroc:>12.4f} {cx:>10.4f} {sc:>12.4f} {tb:>8.4f}")